# Chapter 7 — Let's build GPT

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 7 — Let's build GPT

**Video:** 1h56m · [youtube.com/watch?v=kCc8FmEb1nY](https://www.youtube.com/watch?v=kCc8FmEb1nY) · **The centerpiece.** **Based on:** "Attention Is All You Need" (Vaswani et al., 2017), plus the GPT-2 and GPT-3 papers. **Result:** about 200 lines of code, 10.79 million parameters, producing fake Shakespeare.

*(One correction: in the video Karpathy says GPT-2 is "from 2017 if I recall correctly" [transcript]. GPT-1 was 2018 and GPT-2 was February 2019; 2017 is the transformer paper. A slip of the tongue, noted so the dates line up. [standard])*

### The problem

Every model so far has a fixed, small context and treats it as an undifferentiated blob. Chapter 6's tree combines characters in stages, but the pattern is rigid: position 3 always merges with position 4 regardless of content. What you want is for each position to decide, *based on what it contains*, which earlier positions matter to it.

> **Say it to a six-year-old.** Imagine you are reading a sentence and you get to the word "he." To know who "he" is, your eyes flick back to find the person's name earlier in the sentence. You do not look back at every word equally; you look hardest at the words that help. That flicking back, and choosing what to look at, is the one new idea in this chapter. It is called attention, and it is why computers got good at language.

### Setup

**Run it.**

In [ ]:
import torch
text = open('input.txt', 'r', encoding='utf-8').read()
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"characters: {len(text)} | vocab: {vocab_size}")
print(f"train tokens: {len(train_data)} | val tokens: {len(val_data)}")
print("encode('hi there') ->", encode('hi there'))

**What you should see:**

**Expected output:**

```
characters: 1115394 | vocab: 65
train tokens: 1003854 | val tokens: 111540
encode('hi there') -> [46, 47, 1, 58, 46, 43, 56, 43]
```

[verified]

65 distinct characters, and `hi there` becomes 8 integers because this is character-level. OpenAI's tokenizer would make it 3 [transcript]; that trade-off is Chapter 8.

**The train/validation split is the first 90% and last 10%, not random.** For sequential data you cannot shuffle, because neighbouring characters would end up on both sides of the split and the validation number would be inflated.

### Building attention in five steps

This is the hardest idea in the course, so it comes in stages, each one runnable.

#### Step 1 — what we want

Token 5 should gather information from tokens 1 through 4. It must never see tokens 6 onward, because at generation time those do not exist yet. That restriction is **causal masking**. [standard]

#### Step 2 — the dumbest version that works: averaging

Let each token become the average of itself and all previous tokens.

**Run it.**

In [ ]:
torch.manual_seed(1337)
B, T, C = 1, 4, 2                      # 1 example, 4 tokens, 2 numbers each
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xbow[b, t] = torch.mean(x[b, :t+1], 0)   # average of everything up to and including t
print("running average by loop:", [[round(v, 4) for v in row] for row in xbow[0].tolist()])

**What you should see:**

**Expected output:**

```
running average by loop: [[-2.026, -2.0655], [-1.6157, -1.4889], [-1.4939, -0.7248], [-1.1722, -0.53]]
```

[verified]

Row 1 is just token 1. Row 2 is the average of tokens 1 and 2. Crude, and it does establish communication between positions.

#### Step 3 — averaging is a matrix multiply

**Run it.**

In [ ]:
wei = torch.tril(torch.ones(T, T))     # lower triangular: 1s on and below the diagonal
print(wei)
wei = wei / wei.sum(1, keepdim=True)   # normalize each row to sum to 1
print(wei)
print("same result as the loop:", torch.allclose(xbow, wei @ x))

**What you should see:**

**Expected output:**

```
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])
same result as the loop: True
```

[verified]

**Stare at that matrix, because it is the whole mechanism.** Row 3 says "to build position 3, take one third of each of positions 1, 2, and 3." The zeros in the upper right are what enforce "no peeking ahead." **Any weighted gather over past positions is a matrix multiply with a lower-triangular matrix**, and the double `for` loop is gone, replaced by one operation a GPU does in microseconds.

#### Step 4 — the same thing through softmax

**Run it.**

In [ ]:
import torch.nn.functional as F
tril = torch.tril(torch.ones(T, T))
wei3 = torch.zeros((T, T)).masked_fill(tril == 0, float('-inf'))
print("before softmax:\n", wei3)
wei3 = F.softmax(wei3, dim=-1)
print("after softmax:\n", wei3)
print("identical to averaging:", torch.allclose(wei, wei3))

**What you should see:**

**Expected output:**

```
before softmax:
 tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])
after softmax:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])
identical to averaging: True
```

[verified]

**Why route through softmax when the answer is the same?** Because now those zeros are *scores*, and scores can be computed rather than fixed. `-inf` becomes exactly 0 after softmax, since `exp(-inf) = 0`, so masking is expressed as "give the future a score of negative infinity." Replace the zeros with numbers computed from the data and you have attention.

#### Step 5 — make the weights depend on the content

A flat average treats every previous token as equally relevant, which is wrong: in `The capital of France is`, the word `France` matters far more than `of`.

So every token emits three vectors, each produced by multiplying its own representation by a learned matrix: [standard]

- **Query**: what I am looking for.
- **Key**: what I contain.
- **Value**: what I will hand over if selected.

**Analogy.** A room of people. Each person holds up a label describing themselves (key) and privately holds a note about what they need (query). To decide who to listen to, you compare your note against everyone's label; high match means high attention. Then you receive that person's **value**, which is what they actually say, deliberately kept separate from their label. A person can advertise "I am a date" while contributing "1592."

**Run it.** A single attention head, the real thing:

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16

key   = torch.nn.Linear(C, head_size, bias=False)
query = torch.nn.Linear(C, head_size, bias=False)
value = torch.nn.Linear(C, head_size, bias=False)

k, q, v = key(x), query(x), value(x)
wei = q @ k.transpose(-2, -1) * head_size**-0.5    # scores, scaled
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))    # no peeking ahead
wei = F.softmax(wei, dim=-1)
out = wei @ v

print("scores shape:", tuple(wei.shape), " output shape:", tuple(out.shape))
print("attention row for token 4 (how much it looks at each earlier token):")
print([round(w, 3) for w in wei[0, 3].tolist()])
print("row sums to:", round(wei[0, 3].sum().item(), 5))

**What you should see:**

**Expected output:**

```
scores shape: (4, 8, 8)  output shape: (4, 8, 16)
attention row for token 4 (how much it looks at each earlier token):
[0.3233, 0.2175, 0.2443, 0.2149, 0.0, 0.0, 0.0, 0.0]
row sums to: 1.0
```

[verified]

Read that row carefully, including what it does *not* show. The last four entries are exactly 0.0, so the future is correctly masked. The first four sum to 1 and are uneven, 32% on token 1 versus 21% on token 4, so the weights are content-dependent rather than a flat 0.25 each.

But they are only *slightly* uneven, and that is honest and expected: **this head is untrained**, its query and key matrices are random, so it has not yet learned anything worth attending to. Sharp, interpretable attention patterns are a product of training, not of the mechanism. If you print this same row from a trained model (exercise 1) you will see rows where one position takes 80% and the rest take almost nothing.

**Why the `head_size**-0.5` scaling.** Dot products of `head_size` random numbers have a spread that grows as √head_size. Without dividing it back out, the scores get large, softmax becomes nearly one-hot (section 1.12's "softmax is aggressive"), and each token attends to exactly one other token instead of blending. It is one line and it matters. [transcript]

**Run it.** Watch the failure:

In [ ]:
torch.manual_seed(0)
d = 100
q = torch.randn(1, 1, d)
k = torch.randn(1, 8, d)
raw = q @ k.transpose(-2, -1)
print("raw score spread (std):", round(raw.std().item(), 2), " <- grows as sqrt(d) =", round(d**0.5, 1))
print("unscaled:", [round(w, 4) for w in F.softmax(raw, dim=-1)[0,0].tolist()])
print("scaled:  ", [round(w, 4) for w in F.softmax(raw * d**-0.5, dim=-1)[0,0].tolist()])

**What you should see:**

**Expected output:**

```
raw score spread (std): 12.21  <- grows as sqrt(d) = 10.0
unscaled: [0.0, 0.0, 0.0, 0.9999, 0.0, 0.0, 0.0, 0.0001]
scaled:   [0.026, 0.0273, 0.048, 0.5893, 0.0294, 0.0348, 0.0188, 0.2262]
```

[verified]

The measured spread of 12.21 is close to the predicted √100 = 10, confirming where the scaling factor comes from. Unscaled, softmax put 99.99% on a single token, and a hard selection like that passes almost no gradient to the others, so the head barely learns. Scaled, it puts 59% on its favourite and still meaningfully weighs the alternatives.

**That is self-attention.** "Self" because queries, keys, and values all come from the same sequence.

> **For the PhD in the room.** Attention(Q,K,V) = softmax(QKᵀ/√d_k)V, with the causal mask applied additively pre-softmax. Three things worth being precise about. First, the scaling assumes the query-key entries are roughly zero-mean unit-variance so their dot product has variance d_k, which is why it is √d_k and not d_k. Second, attention is permutation-equivariant, so all positional information comes from the position embeddings added at the input; the mask breaks the symmetry between past and future but not between positions. Third, the O(T²) cost in both compute and memory is the architecture's defining constraint, which is what FlashAttention (Chapter 9) attacks on the memory side by never materializing the T×T matrix, and what linear-attention and state-space models attack on the asymptotic side.

### From one head to a transformer

**Multi-head attention.** Run several attention operations in parallel with separate query, key, and value matrices, then concatenate. Different heads learn different relationships. Karpathy's framing: "it helps to have multiple communication channels because obviously these tokens have a lot to talk about" [transcript]. Four heads took his loss from 2.4 to 2.28.

**Feed-forward layers.** Attention moves information between tokens but does little computation on it. After each attention block, every token independently passes through a small MLP. **Attention is communication; the feed-forward layer is thinking about what you just heard.** The inner layer is 4× wider than the model dimension, a convention from the original paper.

**Residual connections.** Rather than replacing a token's representation, each block computes an adjustment and adds it: `x = x + attention(x)`, then `x = x + feedforward(x)`. From "Deep Residual Learning for Image Recognition," about 2015. [transcript]

Why it works: addition passes gradient through unchanged (section 1.7), so there is a clean, unobstructed path from the loss back to the earliest layers however deep the stack. Without residual connections, deep transformers do not train.

**Analogy.** An express elevator running alongside the stairs. The gradient can always take the elevator.

**LayerNorm.** The same normalization idea as Chapter 4's BatchNorm, computed across the features of a single example instead of across the batch. Because nothing spans examples, the batch-coupling problems disappear along with the running-statistics machinery.

**Run it.** See the difference in one line:

In [ ]:
x = torch.randn(32, 100)
print("BatchNorm normalizes down columns:", tuple(x.mean(0).shape), "-> one statistic per feature")
print("LayerNorm normalizes across rows: ", tuple(x.mean(1).shape), "-> one statistic per example")
ln = torch.nn.LayerNorm(100)
out = ln(x)
print("after LayerNorm, example 0: mean %.4f std %.4f" % (out[0].mean().item(), out[0].std().item()))

**What you should see:**

**Expected output:**

```
BatchNorm normalizes down columns: (100,) -> one statistic per feature
LayerNorm normalizes across rows:  (32,) -> one statistic per example
after LayerNorm, example 0: mean -0.0000 std 1.0050
```

[verified]

**Dropout.** During training, randomly zero 20% of the intermediate values on every forward pass [transcript], and turn it off at evaluation. This stops the network leaning too hard on any single pathway. **Analogy:** a team that rotates who is out sick, so nobody becomes indispensable.

### The full model

**Run it.** The complete architecture, which is the entire model in about 60 lines:

In [ ]:
import torch.nn as nn

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ self.value(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # communicate, then add (residual)
        x = x + self.ffwd(self.ln2(x))   # think, then add
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = self.ln_f(self.blocks(tok_emb + pos_emb))
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])          # crop to the context window
            probs = F.softmax(logits[:, -1, :], dim=-1)     # only the last position matters
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx

**Position embeddings**, the one piece not yet explained: attention has no built-in sense of order, because averaging is the same whatever order you do it in. So a second learned table assigns every *position* (0, 1, 2, ... up to `block_size`) its own vector, and it is added to the token's vector. The model learns what "being third" means.

**Why `x = x + self.sa(self.ln1(x))` and not `self.ln1(x + self.sa(x))`.** Normalizing *before* the sub-block, called pre-norm, keeps the residual path completely clean, so gradient flows from the loss to the first layer without passing through any normalization. The original 2017 paper did it the other way and needed a learning-rate warmup to train at all. [standard]

### Training configuration and results

| Setting | Value |
|---|---|
| Context (block size) | 256 characters, predicting the 257th |
| Batch size | 64 |
| Embedding dimension | 384 |
| Heads | 6, each 64-dimensional (384 ÷ 6) |
| Layers | 6 |
| Dropout | 0.2 |
| Learning rate | 3e-4, AdamW |
| Steps | 5,000 |
| **Parameters** | **10.79 million** [verified] |
| **Training time** | **44 minutes** on one GPU [verified]; ~15 min on an A100 [transcript] |
| **Final validation loss** | **1.4874** [verified]; 1.48 [transcript] |

**What you should see during training.** This is my complete run, start to finish, on a single GPU:

**Expected output:**

```
step     0: train loss 4.2846, val loss 4.2820  [52s]
step   500: train loss 1.8921, val loss 2.0067  [312s]
step  1000: train loss 1.5346, val loss 1.7216  [571s]
step  1500: train loss 1.3953, val loss 1.6085  [831s]
step  2000: train loss 1.3079, val loss 1.5486  [1091s]
step  2500: train loss 1.2519, val loss 1.5177  [1353s]
step  3000: train loss 1.2002, val loss 1.4997  [1616s]
step  3500: train loss 1.1582, val loss 1.4849  [1876s]
step  4000: train loss 1.1199, val loss 1.4837  [2136s]
step  4500: train loss 1.0849, val loss 1.4770  [2396s]
step  4999: train loss 1.0486, val loss 1.4874  [2654s]
```

[verified]

**Final validation loss 1.4874**, against Karpathy's reported **1.48** [transcript]. Independent run, different hardware, same number to three significant figures.

Three things in that table are worth more than the final number.

**The initial loss of 4.2846** against a theoretical 4.1744 for 65 characters (section 1.12) says the initialization is close to correct. That is the first thing to check, before waiting 44 minutes.

**Training and validation separate as it goes.** They start together (4.2846 versus 4.2820), and by the end train is 1.0486 while validation is 1.4874. That widening gap is overfitting appearing in real time, which you now recognize from Chapter 3.

**The best validation loss was at step 4,500, not at the end.** 1.4770 at 4,500 versus 1.4874 at 4,999. The last 500 steps made the model measurably *worse* on data it had not seen while continuing to improve on data it had. In a real project you would keep the step-4,500 checkpoint and discard the rest. This is the practical reason people save checkpoints and use early stopping, and I left it in because a tidier run would have taught you less. [verified]

**And here is what it writes**, sampled from the finished model:

**Expected output:**

```
Lord:
Better therefore flowers, than it so pevenAUe.

LEONTES:
Now, Catesby me,
Treats I, being to extration.

CLAUDIO:
Now, sir, hair been draw that thou occasion;
Offer the lie unto the dearest. If I have
More hopy stood counsel done,
Thou hast colder no greater, but when
We both the sun arms of thy gage brother's death.
Here's great Derformity,
Your good shall womer comes prince: yet,
Who hand that seen the prince wealthould have
Till wear enviroy'd you, and willing success for you.

CLIFFORD:
Now listen the glorious presence, but Son
Go lord, I'll read it from you
At askill my redeign serv
```

[verified]

Look at what it learned from a megabyte of characters, with nobody telling it any of this. Speaker names in capitals followed by a colon and a newline. Real character names, LEONTES from *The Winter's Tale*, CLAUDIO, CLIFFORD, CATESBY, used as speakers rather than scattered. Line lengths that scan like verse. Apostrophes in the right places (`enviroy'd`, `brother's`). Sentences that start with capitals and end with punctuation.

And it is nonsense. `pevenAUe` and `womer` and `hopy` are not words. `Derformity` is one letter from being one. No sentence means anything, and the model has no idea what a Winter's Tale is.

**That gap is the honest summary of what a small language model is**: it has learned the *shape* of the data at every scale below the sentence, and almost nothing above it. Scaling up is what closes the gap, and the architecture does not change. [my read]

**A note on hardware.** "If you don't have a GPU you're not going to be able to reproduce this on a CPU" [transcript]. On a laptop, cut `n_embd` to 64, `n_layer` to 4, `block_size` to 64, and `max_iters` to 2,000. You will get a loss near 1.8 and recognizably worse output, in about ten minutes, and every idea in the chapter will have been demonstrated.

### The scale reality check

Karpathy closes by putting the toy beside the real thing [transcript]:

| | This model | GPT-3 (largest) |
|---|---|---|
| Parameters | 10.79 million | 175 billion |
| Training tokens | ~1 million characters (~300,000 GPT tokens) | 300 billion |
| Ratio | 1 | ~16,000× params, ~1,000,000× data |

And the architecture is the same. That is the point of the lecture, and arguably of the course.

### What is deliberately missing

This builds a **decoder-only** transformer: it reads left to right and generates. The original 2017 paper describes an encoder-decoder for translation, where the decoder also attends to the encoder's output through **cross-attention**. GPT has no encoder, so it is omitted.

It also stops before the stages that turn a language model into ChatGPT: "we did not talk about any of the fine-tuning stages" [transcript]. What this produces is a **pretrained base model**, a document completer. Turning that into an assistant requires supervised fine-tuning on instruction-following examples, then reinforcement learning from human feedback. Those are separate lectures on Karpathy's channel, outside the numbered course.

### Exercises

1. **Print an attention matrix from a trained model** and look at which characters attend to which. Attention after a space often falls on the previous word's start.
2. **Remove the residual connections** (`x = self.sa(self.ln1(x))`) and retrain. Watch training become much worse or fail entirely. This is the most instructive ablation in the chapter.
3. **Remove the position embeddings.** The model still trains, and loses the ability to model word length and line structure. Explain why using the permutation argument.
4. **Set `head_size**-0.5` to `1.0`** and compare the loss curve. Then print the attention rows and confirm they became nearly one-hot.
5. **Train on your own text**: replace `input.txt` with anything at least a megabyte, your own writing, a code repository, song lyrics. This is the moment the course stops being an exercise. [my read]

### Troubleshooting

| Symptom | Cause |
|---|---|
| `CUDA out of memory` | Lower `batch_size`, then `block_size`. Memory scales with batch × block² |
| Loss starts near 4.17 then goes flat | Learning rate too low, or you forgot `optimizer.step()` |
| Loss is `nan` after a few hundred steps | Learning rate too high; 3e-4 is the safe default [transcript] |
| Generation repeats one character forever | You are taking the argmax instead of sampling, or the model is undertrained |
| `RuntimeError: The size of tensor a (256) must match tensor b (8)` | Generation exceeded the context window; crop with `idx[:, -block_size:]` |
| Model trains but generates nothing sensible | Check you are using `logits[:, -1, :]`, the last position only, when generating |

### 30-second version

Let every position in a sequence look back at every earlier position and decide, from the content, which of them matter. Each token broadcasts a label, privately holds a request, matches request against labels, and averages in whatever the best matches offer, with a triangular mask that forbids looking at the future. Stack that with a small per-token computation, an addition shortcut so gradients flow freely, and normalization to keep the scale sane. Six layers of it, 10.79 million parameters, trained for 15 minutes on a megabyte of Shakespeare, writes convincing gibberish in iambic shape. The same architecture at 16,000 times the size is GPT-3.

---